# 05 · End-to-End Walkthrough

This last notebook is not a new topic; it's a short walkthrough tying notebooks 01-04 together into one
run, so you can see how the four workstreams chain into a single pipeline, mirroring `PROPOSAL.md` § 4
as a whole. Each section below is a condensed version of one earlier notebook; go back to that notebook
for the full explanation of any step that isn't clear here.


### Step 0: load the model and cache activations (see notebook 00)

In [1]:
import torch
from transformer_lens import HookedTransformer

torch.manual_seed(0)
model = HookedTransformer.from_pretrained("gpt2", device="cpu")
LAYER = 6

# a small, varied set of short prompts, plenty for a toy demo, not for a real result
PROMPTS = [
    "The capital of France is Paris.",
    "The capital of Germany is Berlin.",
    "Two plus two equals four.",
    "Three plus three equals six.",
    "The sky is blue on a clear day.",
    "The ocean is deep and blue.",
    "The cat sat on the warm mat.",
    "The dog slept on the soft rug.",
    "Water boils at one hundred degrees.",
    "Ice melts at zero degrees.",
    "The sun rises in the east.",
    "The moon orbits the Earth.",
    "She opened the old wooden door.",
    "He closed the heavy metal gate.",
    "The train arrived at the station.",
    "The plane landed on the runway.",
    "Paris is the capital of France.",
    "Berlin is the capital of Germany.",
    "Five minus two equals three.",
    "Ten divided by two equals five.",
]

activations = []
with torch.no_grad():
    for p in PROMPTS:
        _, cache = model.run_with_cache(model.to_tokens(p))
        resid = cache[f"blocks.{LAYER}.hook_resid_post"][0]  # (position, d_model)
        activations.append(resid[1:])  # drop position 0, the prepended BOS token (an outlier "attention sink")

X = torch.cat(activations, dim=0)  # (total_tokens, d_model)
print(f"activation matrix X: {tuple(X.shape)}  (total_tokens, d_model)")


/tmp/ipykernel_3676/3855573954.py:5: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device="cpu")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


activation matrix X: (140, 768)  (total_tokens, d_model)


### Step 1: train a toy SAE (see notebook 01, SAE infra workstream)

In [2]:
import torch.nn as nn

class ToySAE(nn.Module):
    def __init__(self, d_model, dict_size):
        super().__init__()
        self.W_enc = nn.Parameter(torch.randn(d_model, dict_size) * 0.01)
        self.b_enc = nn.Parameter(torch.zeros(dict_size))
        self.W_dec = nn.Parameter(torch.randn(dict_size, d_model) * 0.01)
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        return torch.relu((x - self.b_dec) @ self.W_enc + self.b_enc)

    def decode(self, f):
        return f @ self.W_dec + self.b_dec

sae = ToySAE(d_model=X.shape[1], dict_size=128)
opt = torch.optim.Adam(sae.parameters(), lr=1e-3)
for _ in range(300):
    opt.zero_grad()
    f = sae.encode(X)
    x_hat = sae.decode(f)
    loss = ((x_hat - X) ** 2).mean() + 1e-3 * f.abs().mean()
    loss.backward()
    opt.step()

with torch.no_grad():
    F = sae.encode(X)
    active_frac = (F > 1e-6).float().mean(dim=0)
candidates = ((active_frac > 0.05) & (active_frac < 0.5)).nonzero().squeeze(-1)
feature_idx = candidates[0].item() if len(candidates) > 0 else int(active_frac.argmax())
print(f"trained a toy SAE; picked feature #{feature_idx} (active on {active_frac[feature_idx]:.1%} of tokens) to carry forward")


trained a toy SAE; picked feature #16 (active on 32.1% of tokens) to carry forward


### Step 2: one quick validation check (see notebook 02, feature validation workstream)

In [3]:
with torch.no_grad():
    most_active_row = F[:, feature_idx].argmax().item()
    x = X[most_active_row : most_active_row + 1]
    f_full = sae.encode(x)
    x_hat_full = sae.decode(f_full)

    f_ablated = f_full.clone()
    f_ablated[0, feature_idx] = 0.0
    x_hat_ablated = sae.decode(f_ablated)

    shift = (x_hat_full - x_hat_ablated).norm().item()

print(f"causal-effect check: ablating feature #{feature_idx} shifts the reconstruction by {shift:.4f}")
print("(a full validation would also check specificity, sensitivity, and non-neuron correspondence, see notebook 02)")


causal-effect check: ablating feature #16 shifts the reconstruction by 53.9883
(a full validation would also check specificity, sensitivity, and non-neuron correspondence, see notebook 02)


### Step 3: a causal intervention on the model itself (see notebook 03, causal interventions workstream)

In [4]:
def top_prediction(logits):
    probs = torch.softmax(logits[0, -1, :], dim=-1)
    p, t = probs.max(dim=-1)
    return model.to_string(t.item()), p.item()

prompt_a = "When John and Mary went to the store, John gave a drink to"
prompt_b = "When Mary and John went to the store, Mary gave a drink to"
tokens_a, tokens_b = model.to_tokens(prompt_a), model.to_tokens(prompt_b)

_, cache_a = model.run_with_cache(tokens_a)
word, p = top_prediction(model(tokens_b))
print(f"prompt B baseline: {word!r} (p={p:.3f})")

PATCH_LAYER = 9
def patch_from_a(activation, hook):
    activation[:, -1, :] = cache_a[f"blocks.{PATCH_LAYER}.hook_resid_post"][:, -1, :]
    return activation

patched_logits = model.run_with_hooks(tokens_b, fwd_hooks=[(f"blocks.{PATCH_LAYER}.hook_resid_post", patch_from_a)])
word, p = top_prediction(patched_logits)
print(f"prompt B with prompt A's layer {PATCH_LAYER} residual patched in: {word!r} (p={p:.3f})")


prompt B baseline: ' John' (p=0.411)
prompt B with prompt A's layer 9 residual patched in: ' Mary' (p=0.283)


### Step 4: check against a toy diagnostic (see notebook 04, evaluation / diagnostics workstream)

In [5]:
def entropy(probs):
    probs = probs.clamp_min(1e-12)
    return -(probs * probs.log()).sum().item()

step_confident_correct = torch.tensor([0.95, 0.02, 0.01, 0.01, 0.01])
step_confident_wrong = torch.tensor([0.02, 0.95, 0.01, 0.01, 0.01])

print(f"entropy, confident & correct: {entropy(step_confident_correct):.3f}")
print(f"entropy, confident & WRONG:   {entropy(step_confident_wrong):.3f}  <- indistinguishable from the line above")


entropy, confident & correct: 0.265
entropy, confident & WRONG:   0.265  <- indistinguishable from the line above


## Putting it together

| Step | Workstream | `PROPOSAL.md` section |
|---|---|---|
| 0. Load model, cache activations | (shared foundation) | § 4.2 |
| 1. Train a toy SAE | SAE training / infrastructure | § 4.2 |
| 2. Validate a feature | Feature validation | § 4.3 |
| 3. Intervene causally | Causal interventions | § 4.5 |
| 4. Check against a diagnostic | Evaluation / diagnostics | § 4.4, § 4.7 |

That's the whole loop the real project runs, just at toy scale: cache activations, learn a sparse
decomposition of them, validate that a piece of that decomposition means something, test it causally,
and check the result against a diagnostic designed to catch a specific failure mode. Everything from here
is the same loop, with a real reasoning model, real SAELens training, and real statistical rigor in place
of twenty sentences and 300 epochs.

## Tools reference

Everything in these six notebooks was written in plain PyTorch on purpose, so the mechanism stays visible.
The real pipeline (`PROPOSAL.md` § 4.2, § 4.6, § 8) leans on a small set of actively-maintained,
open-source Python tools instead of hand-rolled code like this. As of writing, here's what each one is
for and where it fits:

| Tool | What it's for | Where it shows up in `PROPOSAL.md` |
|---|---|---|
| **TransformerLens** | Loading models and reading/editing internals via hooks (notebooks 00, 03, 05) | § 4.2, § 8 |
| **SAELens** | Training SAEs at real scale: proper init, dead-feature resampling, LR schedules | § 4.2, § 8 |
| **Circuit Tracer** | Attribution graphs / cross-layer transcoders, the circuit-level fallback | § 4.6 |
| **HuggingFace** | Model and dataset access (GSM8K, ARC-AGI, the reasoning models themselves) | § 4.1, § 4.7, § 8 |
| **Weights & Biases** | Experiment tracking; every tracked run gets tagged with model, layer, dict size, seed | § 8, `CONTRIBUTING.md` |

A couple of things worth knowing before you touch these for real:

- **TransformerLens is migrating its API.** The `DeprecationWarning` you saw in notebook 00 is real:
  `HookedTransformer.from_pretrained` is being superseded by `TransformerBridge.boot_transformers(...)`
  plus `enable_compatibility_mode()`. There's no forced-removal deadline yet, and it's what this whole
  notebook series uses, but expect newer code in the real pipeline to use the bridge API instead.
- **SAELens moved.** It's now maintained under the Decode Research org rather than its original home;
  make sure whatever install instructions or links you're following point at the current repo, not an
  archived fork.
- **If a workstream ever outgrows this machine**, `nnsight`/NDIF (Northeastern) lets you write ordinary
  local PyTorch-style model code that can also execute remotely against much larger models, which is
  worth knowing about even though it isn't part of the current plan.
- **A couple of tools beyond the current stack**, not adopted, but worth knowing exist: EleutherAI's
  `sparsify` (on-the-fly SAE/transcoder training without caching activations to disk first) and `delphi`
  (automated feature explanation/scoring); Neuronpedia, which hosts a public feature explorer and ships
  its own SDK, useful for browsing what other teams' SAEs found without training your own first.
- **For the toy SAE in notebook 01 specifically**, if you want a more thorough from-scratch treatment
  than this notebook gives, Callum McDougall's ARENA SAE exercises build the same idea out further
  against Anthropic's original Toy Models of Superposition setup.

**Where to go next:** read `PROPOSAL.md` § 4 in full, skim the SAELens and TransformerLens docs, and
talk to your workstream lead about what "real" looks like for the piece you'll be working on.
